In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
import optuna
from optuna.pruners import MedianPruner
import boto3
import os
from tqdm import tqdm
import s3fs
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Helper Classes

In [ ]:
class ProphetModel:
    """Facebook Prophet model with Optuna hyperparameter tuning."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.fs = s3fs.S3FileSystem()
        self.s3_client = boto3.client('s3')
        self.df_train = None
        self.df_test = None
        self.df_train_prophet = None
        self.df_test_prophet = None
        self.model = None
        self.forecast = None
        self.dict_best_params = None
    
    def import_data(self):
        """Load train/test splits from S3."""
        str_train_uri = f's3://{self.str_bucket}/02_preprocessing/train_data.csv'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_data.csv'
        
        self.df_train = pd.read_csv(str_train_uri)
        self.df_test = pd.read_csv(str_test_uri)
        
        self.df_train['date'] = pd.to_datetime(self.df_train['date'])
        self.df_test['date'] = pd.to_datetime(self.df_test['date'])
        
        print(f'Loaded train: {len(self.df_train)} samples')
        print(f'Loaded test: {len(self.df_test)} samples')
        
        return self.df_train, self.df_test
    
    def prepare_data(self):
        """Format data for Prophet (ds, y columns)."""
        self.df_train_prophet = self.df_train[['date', 'attrition_rate']].copy()
        self.df_train_prophet.columns = ['ds', 'y']
        
        self.df_test_prophet = self.df_test[['date', 'attrition_rate']].copy()
        self.df_test_prophet.columns = ['ds', 'y']
        
        print(f'Prepared data for Prophet:')
        print(f'  Train: {len(self.df_train_prophet)} samples')
        print(f'  Test: {len(self.df_test_prophet)} samples')
        
        return self.df_train_prophet, self.df_test_prophet
    
    def objective(self, trial):
        """Optuna objective function for hyperparameter tuning."""
        flt_changepoint_prior_scale = trial.suggest_float('changepoint_prior_scale', 0.001, 0.5)
        flt_seasonality_prior_scale = trial.suggest_float('seasonality_prior_scale', 0.01, 10.0)
        str_seasonality_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
        
        try:
            model = Prophet(
                changepoint_prior_scale=flt_changepoint_prior_scale,
                seasonality_prior_scale=flt_seasonality_prior_scale,
                seasonality_mode=str_seasonality_mode,
                yearly_seasonality=True,
                interval_width=0.95
            )
            
            model.fit(self.df_train_prophet)
            
            # Evaluate on test set
            df_forecast_test = model.predict(self.df_test_prophet[['ds']])
            arr_actual = self.df_test_prophet['y'].values
            arr_pred = df_forecast_test['yhat'].values
            
            flt_mape = np.mean(np.abs((arr_actual - arr_pred) / arr_actual))
            
            return flt_mape
        except:
            return float('inf')
    
    def tune_model(self, int_n_trials=20):
        """Optuna hyperparameter tuning."""
        print(f'\nTuning Prophet with Optuna ({int_n_trials} trials)...')
        
        study = optuna.create_study(
            direction='minimize',
            pruner=MedianPruner()
        )
        
        study.optimize(self.objective, n_trials=int_n_trials, show_progress_bar=True)
        
        self.dict_best_params = study.best_params
        
        print(f'\nBest parameters:')
        for str_key, str_val in self.dict_best_params.items():
            print(f'  {str_key}: {str_val}')
        print(f'Best MAPE: {study.best_value:.4f}')
        
        return study
    
    def fit_model(self):
        """Fit Prophet with best parameters."""
        print(f'\nFitting Prophet with best parameters...')
        
        self.model = Prophet(
            changepoint_prior_scale=self.dict_best_params['changepoint_prior_scale'],
            seasonality_prior_scale=self.dict_best_params['seasonality_prior_scale'],
            seasonality_mode=self.dict_best_params['seasonality_mode'],
            yearly_seasonality=True,
            interval_width=0.95
        )
        
        self.model.fit(self.df_train_prophet)
        print('Model fitted successfully!')
        
        return self.model
    
    def forecast(self, int_periods=12):
        """Generate forecast."""
        df_future = self.model.make_future_dataframe(periods=int_periods, freq='MS')
        self.forecast = self.model.predict(df_future)
        
        # Extract forecast for future periods
        df_forecast = self.forecast[self.forecast['ds'] > self.df_train_prophet['ds'].max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
        df_forecast.columns = ['date', 'forecast', 'ci_lower', 'ci_upper']
        
        print(f'\nGenerated {int_periods}-step forecast')
        print(df_forecast.head(10))
        
        return df_forecast
    
    def plot_forecast(self, df_forecast):
        """Plot forecast using Prophet's built-in plotting."""
        # Prophet's default plot
        fig = self.model.plot(self.forecast)
        fig.suptitle('Prophet Forecast', fontsize=14, fontweight='bold', y=1.00)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/13_prophet_forecast_default.png'
        fig.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved default plot to {str_path}')
        plt.show()
        
        # Custom overlay plot
        fig, ax = plt.subplots(figsize=(14, 7))
        
        # Training data
        ax.plot(self.df_train_prophet['ds'], self.df_train_prophet['y'], linewidth=2, 
               label='Training Data', marker='o', markersize=4)
        
        # Test data
        ax.plot(self.df_test_prophet['ds'], self.df_test_prophet['y'], linewidth=2, 
               label='Test Data', marker='s', markersize=4, color='#ff7f0e')
        
        # Forecast
        ax.plot(df_forecast['date'], df_forecast['forecast'], linewidth=2.5, 
               label='Prophet Forecast', marker='^', markersize=5, color='#d62728')
        
        # Confidence intervals
        ax.fill_between(df_forecast['date'], df_forecast['ci_lower'], df_forecast['ci_upper'],
                        alpha=0.3, color='#d62728', label='95% Confidence Interval')
        
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Prophet Forecast vs Actual', fontsize=13, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/14_prophet_forecast_custom.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved custom plot to {str_path}')
        plt.show()
    
    def plot_components(self):
        """Plot trend and seasonality components."""
        fig = self.model.plot_components(self.forecast, include_legend=True)
        fig.suptitle('Prophet Components (Trend and Seasonality)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/15_prophet_components.png'
        fig.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved components plot to {str_path}')
        plt.show()
    
    def evaluate(self, df_forecast):
        """Calculate RMSE, MAE, MAPE on test set."""
        arr_actual = self.df_test_prophet['y'].values
        arr_pred = df_forecast['forecast'].values[:len(arr_actual)]
        
        flt_rmse = np.sqrt(np.mean((arr_actual - arr_pred) ** 2))
        flt_mae = np.mean(np.abs(arr_actual - arr_pred))
        flt_mape = np.mean(np.abs((arr_actual - arr_pred) / arr_actual)) * 100
        
        dict_metrics = {
            'RMSE': flt_rmse,
            'MAE': flt_mae,
            'MAPE': flt_mape
        }
        
        print(f'\nTest Set Evaluation:')
        print(f'  RMSE: {flt_rmse:.6f}')
        print(f'  MAE: {flt_mae:.6f}')
        print(f'  MAPE: {flt_mape:.2f}%')
        
        return dict_metrics
    
    def save_model(self):
        """Serialize model to disk and S3."""
        str_model_path = f'{self.str_dirname_output}/prophet_model.pkl'
        joblib.dump(self.model, str_model_path)
        
        try:
            self.s3_client.upload_file(
                str_model_path,
                self.str_bucket,
                '04_prophet/prophet_model.pkl'
            )
            print(f'Saved model to {str_model_path} and S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')

## Constants

In [ ]:
str_bucket = 'time-series-forecasting-demo-repo'
str_task = 'employee_attrition_forecasting'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

print(f'Output directory: {str_dirname_output}')

## Load and Prepare Data

In [ ]:
prophet_model = ProphetModel(str_bucket, str_dirname_output)
df_train, df_test = prophet_model.import_data()
df_train_prophet, df_test_prophet = prophet_model.prepare_data()

## Hyperparameter Tuning with Optuna

In [ ]:
study = prophet_model.tune_model(int_n_trials=20)

## Fit Prophet Model

In [ ]:
model = prophet_model.fit_model()

## Generate Forecast

In [ ]:
df_forecast = prophet_model.forecast(int_periods=12)

## Plot Forecast

In [ ]:
prophet_model.plot_forecast(df_forecast)

## Plot Components

In [ ]:
prophet_model.plot_components()

## Evaluate Model

In [ ]:
dict_metrics = prophet_model.evaluate(df_forecast)

## Save Model

In [ ]:
prophet_model.save_model()

## Model Summary

In [ ]:
print(f'\n=== PROPHET MODEL SUMMARY ===')
print(f'Changepoint Prior Scale: {prophet_model.dict_best_params["changepoint_prior_scale"]:.4f}')
print(f'Seasonality Prior Scale: {prophet_model.dict_best_params["seasonality_prior_scale"]:.4f}')
print(f'Seasonality Mode: {prophet_model.dict_best_params["seasonality_mode"]}')
print(f'Test RMSE: {dict_metrics["RMSE"]:.6f}')
print(f'Test MAE: {dict_metrics["MAE"]:.6f}')
print(f'Test MAPE: {dict_metrics["MAPE"]:.2f}%')
print(f'\nModel ready for production deployment!')